# Train / Validation / Test Splits & Data Leakage Review

This Phase 1 review focuses on:

- roles of training, validation, and test sets
- preprocessing leakage
- `fit()` vs `transform()`
- target / future leakage
- temporal and group-aware splitting
- stratification
- overfitting vs underfitting
- validation-set overfitting
- the role of the final test set
- retraining on training + validation after model selection


# Vocabulary Primer

## Target

The **target** is the real outcome the model is trying to predict, often written as $y$.

Examples:
- spam classifier → target = spam / not spam
- disease classifier → target = disease / no disease
- house-price model → target = true sale price
- country classifier → target = true country label

A **feature** is an input used to predict the target.

Example:

```text
features: alcohol %, grape type, acidity, province
target: country
```

So yes: the target is the ground-truth reality we hope the prediction matches.


# Training Time vs Inference Time

**Training time** is when the model learns from training data and updates learned parameters such as neural-network weights.

**Inference time** is after training, when the trained model is used to make predictions on new data.

Examples of inference:
- classifying a new image
- scoring a new transaction for fraud
- making a prediction inside an app or API

At inference time, model weights normally do not change.

For dropout:
- training time → dropout is active
- inference time → dropout is normally disabled

Dropout randomly sets some **activations** to zero during training. It does not normally delete entire layers or permanently remove weights.


# What Does “Parameter” Mean Here?

The word **parameter** can mean different things.

## Population parameter
From statistics:
- population mean $\mu$
- population SD $\sigma$
- population proportion $p$

## Model parameter
Values learned by the model:
- neural-network weights
- biases
- regression coefficients

## Preprocessing parameter
Values learned by preprocessing:
- scaler → feature means and SDs
- imputer → medians / means / modes used to fill missing values
- PCA → principal directions/components

So when we say “fit the scaler,” we mean learning the scaler's preprocessing values from training data, not learning neural-network weights or population parameters.


# Scaler, Imputer, and PCA

## Scaler
A **scaler** changes the numerical scale of features.

For standardization:

$$
z=\frac{x-\mu_{train}}{\sigma_{train}}
$$

The scaler **fits** by learning a mean and SD for each feature from training data, then **transforms** train/validation/test using those same values.

## Imputer
An **imputer** fills in missing values.

Example:
```text
income = [40k, 50k, missing, 60k]
```

A median imputer may learn:
```text
training median = 50k
```

and use 50k to fill missing values everywhere.

If that median were calculated using the test set too, that would be preprocessing leakage.

## PCA
**PCA** = Principal Component Analysis.

PCA is a dimensionality-reduction method. It learns new directions through feature space that capture as much variation as possible.

Conceptually:

```text
original features
      ↓
PCA learns important directions from TRAINING data
      ↓
new compressed features ("principal components")
```

Those learned directions are PCA's preprocessing parameters.


# Fit vs Transform — Plain English

## Fit
**Fit = learn something from data.**

Examples:
- scaler learns mean and SD
- imputer learns a median
- PCA learns principal directions
- model learns weights

## Transform
**Transform = apply what was already learned.**

Example:

```text
TRAIN:
fit scaler → learn mean=40, SD=10
transform training data

VALIDATION:
do not learn a new mean/SD
transform using mean=40, SD=10

TEST:
do not learn a new mean/SD
transform using mean=40, SD=10
```

Core rule:

**Split first → fit preprocessing on training data → transform validation/test using the fitted training preprocessing.**


# 1. Core Split

## Training set
Used to fit learned model parameters such as weights, coefficients, or tree splits.

## Validation set
Used for model-development choices such as:

- hyperparameters
- architecture
- feature choices
- thresholds
- early stopping
- choosing between candidate models

## Test set
Used for a final independent evaluation after model-development decisions are finished.

A useful mental model:

$$
\boxed{\text{Train} \rightarrow \text{Learn}}
$$

$$
\boxed{\text{Validation} \rightarrow \text{Choose}}
$$

$$
\boxed{\text{Test} \rightarrow \text{Evaluate}}
$$


# 2. Why Train + Test Alone Is Often Not Enough

If you repeatedly compare models on the test set and use those results to make decisions, the test set becomes part of model selection.

That means it no longer provides a clean independent estimate of generalization.

Typical workflow:

$$
\text{Train} \rightarrow \text{Validation decisions} \rightarrow \text{One final test}
$$


# 3. Validation Overfitting

Repeatedly redesigning a model based on validation results can make the overall development process specialize to that validation set.

This can happen even though the model's weights are never directly trained on validation examples.

The final test set therefore evaluates the **whole model-development process**.

If:

$$
\text{Validation performance} \approx \text{Test performance}
$$

that supports generalization.

If:

$$
\text{Validation performance} \gg \text{Test performance}
$$

the development process may have overfit to validation data.


# 4. Data Leakage

**Data leakage** occurs when information that should not be available during model development or at real prediction time influences the model, features, preprocessing, or model-selection process.

Common forms include:

- preprocessing leakage
- target leakage
- future leakage
- group leakage
- duplicate leakage
- test-set leakage


# 5. Preprocessing Leakage

Suppose standardization uses:

$$
z=\frac{x-\mu}{\sigma}
$$

Incorrect workflow:

1. calculate $\mu$ and $\sigma$ using the entire dataset
2. standardize everything
3. split into train / validation / test

This leaks information from validation/test data into preprocessing.

Correct workflow:

1. split first
2. calculate $\mu_{train}$ and $\sigma_{train}$ from training data
3. transform all sets using those training-derived values

$$
z_{train}=\frac{x-\mu_{train}}{\sigma_{train}}
$$

$$
z_{val}=\frac{x-\mu_{train}}{\sigma_{train}}
$$

$$
z_{test}=\frac{x-\mu_{train}}{\sigma_{train}}
$$

Do **not** fit separate scalers on validation and test data.


# 6. `fit()` vs `transform()`

## `fit()`
Learn parameters from data.

Examples:

- StandardScaler learns mean and SD
- imputation may learn medians
- PCA learns directions
- feature selection learns which features to keep

## `transform()`
Apply already-learned parameters.

Core rule:

$$
\boxed{\text{fit on training data}}
$$

then:

$$
\boxed{\text{transform train, validation, and test}}
$$


In [ ]:
# Typical scikit-learn preprocessing pattern

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)


# 7. Pipelines Help Prevent Leakage

A scikit-learn `Pipeline` bundles preprocessing and model fitting together.

This is especially useful with cross-validation because each training fold gets its own fitted preprocessing.

Conceptually:

$$
\text{training data}
\rightarrow
\text{fit preprocessing}
\rightarrow
\text{transform}
\rightarrow
\text{fit model}
$$


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression())
])

pipeline.fit(X_train, y_train)
val_score = pipeline.score(X_val, y_val)


# 8. Target Leakage

**Target leakage** occurs when a feature reveals information about the target that would not legitimately be available when making the prediction.

Example:

Goal: predict at hospital discharge whether a patient will later be readmitted.

Feature:

```text
days_until_readmission
```

This depends on an outcome that happens after prediction time.

Useful question:

> **Would this feature exist at the exact moment the real prediction must be made?**

If not, it may be leakage.


# 9. Future Leakage and Temporal Splits

For forecasting, random shuffling can leak future information into training.

Example task:

> Predict next month's sales.

Better evaluation structure:

- 2020–2024 → training
- 2025 → validation
- 2026 → test

The evaluation should imitate deployment:

$$
\boxed{\text{past} \rightarrow \text{future}}
$$


In [ ]:
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)

for train_idx, val_idx in tscv.split(X):
    X_train_fold = X.iloc[train_idx]
    X_val_fold = X.iloc[val_idx]


# 10. Group Leakage

If the same person, device, household, or other entity appears in multiple splits, the model may partly recognize the entity instead of generalizing.

Example:

- some records from Patient A in training
- other records from Patient A in test

This can inflate test performance.

Group-aware splitting keeps all records from one group together.


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=patient_ids)
)


# 11. Duplicate Leakage

Exact or near-duplicate examples across splits can make evaluation look better than true generalization.

Possible examples:

- duplicate records
- multiple crops of the same image
- near-identical text
- repeated measurements from one subject

Always consider duplicates before trusting metrics.


# 12. Overfitting

Example:

$$
\text{Train Accuracy}=99\%
$$

$$
\text{Validation Accuracy}=78\%
$$

This large gap suggests overfitting.

Possible responses:

- more data
- regularization
- dropout
- weight decay
- early stopping
- simpler model
- appropriate data augmentation
- investigate distribution mismatch
- check duplicates/leakage

Important:

The goal is not to intentionally lower training accuracy. The goal is to improve generalization.


# 13. Underfitting / High Bias

Example:

$$
\text{Train Accuracy}=71\%
$$

$$
\text{Validation Accuracy}=70\%
$$

The small gap means there is little evidence of overfitting.

But both scores may still be poor.

If the task should permit better performance, this may indicate:

$$
\boxed{\text{underfitting / high bias}}
$$

Possible causes:

- model too simple
- insufficient training
- weak features
- excessive regularization
- optimization problems

Key lesson:

$$
\boxed{\text{small train-validation gap} \neq \text{good model}}
$$


# 14. Dropout

Dropout does **not** normally mean deleting entire layers.

During training, dropout temporarily sets a random subset of activations to zero.

This can reduce overfitting by preventing the network from depending too heavily on particular pathways.

At inference time, dropout is normally disabled.


# 15. Stratified Splitting

When a class is rare, random splitting may create unstable class proportions.

A **stratified split** tries to preserve approximately the same class ratio across splits.

Example:

| Split | Fraud rate |
|---|---:|
| Full dataset | 0.5% |
| Train | ≈0.5% |
| Validation | ≈0.5% |
| Test | ≈0.5% |


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


# 16. Feature Selection Can Leak

If you use the full dataset to decide which features correlate most strongly with the target, then the test set has already influenced model development.

Correct idea:

$$
\boxed{\text{split first}}
$$

Then learn feature-selection decisions from training data only.


# 17. Retraining on Training + Validation

After:

1. model selection is finished
2. the final configuration is chosen
3. the final test evaluation is recorded

it can be reasonable to retrain the chosen configuration using:

$$
\boxed{\text{training + validation}}
$$

This gives the production model more data.

However, the original test result strictly corresponds to the development procedure before that final retraining.

Do not repeatedly evaluate the retrained model on the same test set and continue changing the model based on those results.

If a completely fresh unbiased evaluation of the newly retrained artifact is required, another untouched holdout or appropriate evaluation design is needed.


# 18. Vocabulary Summary

| Term | Meaning |
|---|---|
| Training set | Data used to fit learned model parameters |
| Validation set | Data used for model-development choices |
| Test set | Final independent evaluation data |
| Fit | Learn parameters from data |
| Transform | Apply already-learned parameters |
| Data leakage | Unavailable or inappropriate information influences model development |
| Preprocessing leakage | Validation/test information influences preprocessing |
| Target leakage | Features reveal the target |
| Future leakage | Future information influences prediction of the past/present |
| Group leakage | Related examples from the same entity cross splits |
| Stratification | Preserve class proportions across splits |
| Overfitting | Training performance is much stronger than unseen-data performance |
| Underfitting | Model fails to capture enough signal even on training data |
| High bias | Common description of underfitting |
| High variance | Often associated with overfitting |
| Validation overfitting | Repeated decisions specialize to validation data |
| Holdout set | Data reserved for independent evaluation |
| Temporal split | Split based on chronology |


# 19. Practical Checklist

Before trusting model evaluation, ask:

### Splitting
- Was the split done before fitting preprocessing?
- Does the split imitate real deployment?
- Is chronology important?
- Are there groups that must stay together?
- Is stratification needed?

### Features
- Would every feature exist at prediction time?
- Could any feature reveal the target?
- Was feature selection learned from training data only?

### Preprocessing
- Were scaler/imputer/PCA parameters learned from training data only?
- Are validation/test transformed using training-derived parameters?

### Evaluation
- Was validation used for tuning?
- Was test excluded from model-development decisions?
- Are both train and validation scores actually strong?
- Is there a suspicious train/validation gap?
- Are duplicates present across splits?


# 20. Targeted Closed-Notes Review

After reviewing this notebook, answer these without looking back.

## Preprocessing and leakage

1. Why is fitting `StandardScaler` on the full dataset before splitting leakage?
2. Why should validation/test use the training-set mean and SD rather than their own?
3. Explain `fit()` vs `transform()`.
4. If missing values are filled using the median calculated from the entire dataset before splitting, is that leakage?

## Target and future leakage

5. You are predicting whether a student will pass a course. One feature is their final exam grade. What is wrong?
6. You are predicting tomorrow's stock movement but one feature uses next week's average price. What type of leakage is this?
7. Give one original example of a feature available during dataset construction but unavailable at real prediction time.

## Splitting strategy

8. Why is random splitting often inappropriate for time-series forecasting?
9. A medical dataset has 20 rows per patient. What splitting strategy should you consider?
10. A dataset has 1% positive cases. What does stratification try to preserve?

## Diagnosing model behavior

11. Train accuracy = 98%, validation accuracy = 74%. What does this suggest?
12. Train accuracy = 63%, validation accuracy = 62%. Why does the small gap not automatically mean the model is good?
13. Give two possible ways to address overfitting.
14. Give two possible causes of underfitting.

## Validation and test usage

15. How can a human overfit to the validation set?
16. Why evaluate once on an untouched test set after model selection?
17. What goes wrong if you repeatedly choose models based on test performance?
18. Why might you retrain the final configuration on training + validation?
19. After retraining, why should you avoid repeatedly consulting the same test set while making further changes?

## Final synthesis

20. Define data leakage in one sentence.
21. Name four distinct forms of leakage.
22. Complete this sentence:

**Split first → _____ preprocessing on training data → _____ validation/test data**

23. In one sentence each, explain the role of:
- training set
- validation set
- test set
